# ChuckleNet: Scale to 555 Videos (v17 - ALL CRITICAL FIXES)

**Strategy:**
1. Use existing WavLM embeddings from `wavlm_utterance_safe` (555 videos)
2. Extract missing prosody (21-dim) for ALL videos
3. Pre-convert M4A → WAV for faster I/O
4. **Checkpoint every 50 videos**
5. **Proper 2-class CrossEntropyLoss**

**v17 CRITICAL FIXES (from agent council):**
- ✅ Added gradient clipping (max_norm=1.0) - PREVENTS DIVERGENCE
- ✅ Fixed class weights [1.0, 2.5] - MATCHES VALIDATED PIPELINE
- ✅ Fixed batch size 256 - MATCHES VALIDATED PIPELINE
- ✅ Reload checkpoints into memory on restart
- ✅ Added warnings for silent match failures

**Time estimate:** 1-2 hours (vs 30+ hours before)

In [ ]:
# Step 1: Mount Drive
from google.colab import drive
drive.mount('/content/gdrive')
print('✅ Drive mounted!')

In [ ]:
# Step 2: Install dependencies
!pip install -q transformers librosa scikit-learn soundfile pandas
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ Dependencies installed!')

In [ ]:
# PRE-CHECK: Verify JSON files before loading
from pathlib import Path
import json

WAVLM_DIR = Path('/content/gdrive/MyDrive/wavlm_utterance_safe')

print('🔍 Pre-checking JSON files...')
bad_files = []
good_files = []

for json_file in WAVLM_DIR.glob('*.json'):
    try:
        with open(json_file) as f:
            data = json.load(f)
        if isinstance(data, dict) and 'embeddings' in data:
            good_files.append(json_file.stem)
        else:
            bad_files.append(json_file.name)
    except Exception as e:
        bad_files.append(f"{json_file.name} (error: {e})")

print(f'✅ Good files: {len(good_files)}')
print(f'❌ Bad files: {len(bad_files)}')
if bad_files:
    print(f'   First 5 bad: {bad_files[:5]}')
    print('   Will skip bad files during loading...')

# Store good files list for later use
GOOD_VIDEO_IDS = set(good_files)
print(f'   Will load {len(GOOD_VIDEO_IDS)} videos')


## Step 3: Load existing WavLM embeddings from wavlm_utterance_safe

In [ ]:
import json
from pathlib import Path
import numpy as np
import time

WAVLM_DIR = Path('/content/gdrive/MyDrive/wavlm_utterance_safe')

print('📗 Loading existing WavLM embeddings...')
wavlm_data = {}  # video_id -> list of {start, end, embedding}

for json_file in WAVLM_DIR.glob('*.json'):
    vid = json_file.stem
    if vid not in GOOD_VIDEO_IDS:
        continue  # Skip bad files
    vid = json_file.stem
    with open(json_file) as f:
        data = json.load(f)
    wavlm_data[vid] = data['embeddings']

print(f'✅ Loaded WavLM for {len(wavlm_data)} videos')

total_utterances = sum(len(v) for v in wavlm_data.values())
print(f'   Total utterances: {total_utterances}')

## Step 4: Check for existing prosody data (skip if already extracted)

In [ ]:
# Check which videos already have prosody extracted
PROSODY_CHECKPOINT = Path('/content/gdrive/MyDrive/wavlm_prosody_checkpoints')
PROSODY_CHECKPOINT.mkdir(exist_ok=True)

# Get list of already-extracted videos
extracted_videos = set()
for ckpt_file in PROSODY_CHECKPOINT.glob('prosody_*.npz'):
    vid = ckpt_file.stem.replace('prosody_', '')
    extracted_videos.add(vid)

print(f'📊 Prosody extraction status:')
print(f'   Already extracted: {len(extracted_videos)} videos')
print(f'   Remaining: {len(wavlm_data) - len(extracted_videos)} videos')
print(f'   Total: {len(wavlm_data)} videos')

## Step 5: Load utterances to get labels

In [ ]:
import subprocess
from pathlib import Path

UTT_PATH = Path('/content/gdrive/MyDrive/utterances_clean.jsonl')
if not UTT_PATH.exists():
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True, capture_output=True)
    subprocess.run(['gdown', '--fuzzy', '-O', str(UTT_PATH),
        'https://drive.google.com/file/d/1cuhs6mh-r9Spzq9cTDG8AidT53DLsALn/view'],
        check=True, capture_output=True)

print('�📖 Loading utterances...')
utterances = []
with open(UTT_PATH) as f:
    for line in f:
        utterances.append(json.loads(line.strip()))

print(f'✅ Loaded {len(utterances)} utterances')

# Create lookup: (video_id, start, end) -> label
label_lookup = {}
for u in utterances:
    key = (u['video_id'], round(u['start'], 2), round(u['end'], 2))
    label_lookup[key] = u['label']

print(f'   Labeled: {len(label_lookup)}')

## Step 6: Match WavLM embeddings with labels

In [ ]:
# Match embeddings with labels
matched = 0
unmatched = 0

for vid, embeddings in wavlm_data.items():
    for emb in embeddings:
        key = (vid, round(emb['start'], 2), round(emb['end'], 2))
        if key in label_lookup:
            emb['label'] = label_lookup[key]
            matched += 1
        else:
            emb['label'] = 0
            unmatched += 1

print(f'✅ Matched: {matched} utterances with labels')
print(f'⚠️ Unmatched: {unmatched}')

# Stats
pos = sum(1 for v in wavlm_data.values() for e in v if e.get('label') == 1)
neg = sum(1 for v in wavlm_data.values() for e in v if e.get('label') == 0)
print(f'   Positive: {pos} ({pos/(pos+neg)*100:.1f}%)')
print(f'   Negative: {neg} ({neg/(pos+neg)*100:.1f}%)')

## Step 7: Split into train/val/test (video-level - NO LEAKAGE)

In [ ]:
import random

video_ids = list(wavlm_data.keys())
random.seed(42)
random.shuffle(video_ids)

n = len(video_ids)
val_vids = set(video_ids[:int(n*0.1)])
test_vids = set(video_ids[int(n*0.1):int(n*0.2)])

train_data = []
val_data = []
test_data = []

for vid in video_ids:
    if vid in val_vids:
        val_data.extend(wavlm_data[vid])
    elif vid in test_vids:
        test_data.extend(wavlm_data[vid])
    else:
        train_data.extend(wavlm_data[vid])

print(f'📊 Split (video-level):')
print(f'   Train: {len(train_data)} utterances ({n - len(val_vids) - len(test_vids)} videos)')
print(f'   Val: {len(val_data)} utterances ({len(val_vids)} videos)')
print(f'   Test: {len(test_data)} utterances ({len(test_vids)} videos)')

## Step 8: Pre-convert M4A to WAV (OPTIMIZATION)

In [ ]:
import subprocess
from pathlib import Path

BASE = Path('/content/gdrive/MyDrive')
WAV_DIR = BASE / 'chuckle_audio_wav_v2'
WAV_DIR.mkdir(exist_ok=True)

# Find M4A files
m4a_files = {}
for folder in ['chuckle_audio', 'chuckle_audio_all/audio', 'chuckle_audio_all/audio_final', 'chuckle_audio_all/audio_new']:
    audio_dir = BASE / folder
    if audio_dir.exists():
        for p in audio_dir.glob('*.m4a'):
            m4a_files[p.stem] = p

print(f'📦 Found {len(m4a_files)} M4A files to convert')

# Convert to WAV
converted = 0
skipped = 0

for i, (stem, m4a_path) in enumerate(m4a_files.items()):
    wav_path = WAV_DIR / (stem + '.wav')
    
    if wav_path.exists():
        skipped += 1
        continue
    
    subprocess.run([
        'ffmpeg', '-y', '-i', str(m4a_path),
        '-ar', '16000', '-ac', '1',
        '-loglevel', 'error',
        str(wav_path)
    ], check=False, capture_output=True)
    
    converted += 1
    
    if (i + 1) % 50 == 0:
        print(f'  Progress: {i+1}/{len(m4a_files)} | {converted} converted, {skipped} exist')

print(f'\n✅ Converted {converted} | Skipped {skipped}')

## Step 9: Extract prosody for each video (WITH CHECKPOINT - FIXED!)

In [ ]:
import librosa
import numpy as np
import soundfile as sf
import time

SR = 16000

def extract_prosody_21dim(y, sr):
    """Extract 21 prosody features."""
    features = []
    
    # F0 (pitch) - 5 dims
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        features.extend([
            np.mean(f0_clean) if len(f0_clean) > 0 else 0,
            np.std(f0_clean) if len(f0_clean) > 0 else 0,
            np.max(f0_clean) if len(f0_clean) > 0 else 0,
            np.min(f0_clean) if len(f0_clean) > 0 else 0,
            np.sum(voiced_flag) / len(voiced_flag) if len(voiced_flag) > 0 else 0
        ])
    except:
        features.extend([0]*5)
    
    # Energy - 5 dims
    rms = librosa.feature.rms(y=y)[0]
    features.extend([
        np.mean(rms), np.std(rms), np.max(rms), np.min(rms),
        np.max(rms) - np.min(rms)
    ])
    
    # Duration - 2 dims
    features.extend([
        len(y) / sr,
        len(y) / sr / (np.sum(rms > np.mean(rms)) + 1)
    ])
    
    # Spectral - 5 dims
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    spec_flat = librosa.feature.spectral_flatness(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features.extend([
        np.mean(spec_cent), np.mean(spec_bw), np.mean(spec_flat),
        np.mean(zcr), np.std(zcr)
    ])
    
    # Voice quality - 4 dims
    try:
        hnr = librosa.effects.hpss(y)[1]
        hnr_val = np.mean(hnr) / (np.mean(np.abs(y)) + 1e-8)
    except:
        hnr_val = 0
    features.extend([hnr_val, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y))])
    
    return np.array(features, dtype=np.float32)

def get_audio_path(vid):
    # Try WAV first (converted)
    wav_path = WAV_DIR / f'{vid}.wav'
    if wav_path.exists():
        return str(wav_path)
    
    # Try original locations
    for folder in ['chuckle_audio', 'chuckle_audio_all/audio', 
                  'chuckle_audio_all/audio_final', 'chuckle_audio_all/audio_new']:
        audio_dir = BASE / folder
        if audio_dir.exists():
            for ext in ['.wav', '.mp3', '.m4a']:
                p = audio_dir / f'{vid}{ext}'
                if p.exists() and not p.name.endswith('.part'):
                    return str(p)
    return None

def save_checkpoint(vid, prosody_array):
    """Save prosody for a single video."""
    np.savez_compressed(PROSODY_CHECKPOINT / f'prosody_{vid}.npz', 
                       prosody=prosody_array,
                       video_id=vid)

def load_checkpoint(vid):
    """Load prosody for a single video."""
    ckpt_file = PROSODY_CHECKPOINT / f'prosody_{vid}.npz'
    if ckpt_file.exists():
        data = np.load(ckpt_file)
        return data['prosody']
    return None

In [ ]:
# Extract prosody for all videos WITH CHECKPOINT (FIXED!)
print('🔍 Extracting prosody for all videos...')
t0 = time.time()

prosody_data = {}  # video_id -> list of prosody arrays
failed = []
skipped_already = 0

for i, vid in enumerate(wavlm_data.keys()):
    # Skip if already extracted
    if vid in extracted_videos:
        prosody = load_checkpoint(vid)
        if prosody is not None:
            prosody_data[vid] = prosody.tolist()
            skipped_already += 1
            continue
    
    audio_path = get_audio_path(vid)
    
    if not audio_path:
        failed.append(vid)
        prosody_data[vid] = [np.zeros(21, dtype=np.float32).tolist()] * len(wavlm_data[vid])
        continue
    
    try:
        # Load audio
        if audio_path.endswith('.wav'):
            y, sr = sf.read(audio_path, dtype='float32')
        else:
            y, sr = librosa.load(audio_path, sr=SR, mono=True)
        
        if len(y.shape) > 1:
            y = y.mean(axis=1)
        
        if sr != SR:
            y = librosa.resample(y, orig_sr=sr, target_sr=SR)
        
        # Extract prosody for each utterance
        video_prosody = []
        for emb in wavlm_data[vid]:
            start_s = emb['start']
            end_s = emb['end']
            
            start_sample = int(start_s * SR)
            end_sample = int(end_s * SR)
            
            if end_sample > len(y):
                end_sample = len(y)
            
            y_slice = y[start_sample:end_sample]
            
            if len(y_slice) < SR * 0.1:
                video_prosody.append(np.zeros(21, dtype=np.float32))
            else:
                prosody = extract_prosody_21dim(y_slice, SR)
                video_prosody.append(prosody)
        
        prosody_array = np.array(video_prosody, dtype=np.float32)
        prosody_data[vid] = prosody_array.tolist()
        
        # Save checkpoint for this video
        save_checkpoint(vid, prosody_array)
        
    except Exception as e:
        failed.append(vid)
        prosody_data[vid] = [np.zeros(21, dtype=np.float32).tolist()] * len(wavlm_data[vid])
    
    # Progress update every 50 videos
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (len(wavlm_data) - i - 1)
        print(f'📊 {i+1}/{len(wavlm_data)} | ETA: {eta/60:.1f} min | Failed: {len(failed)} | Skipped: {skipped_already}')

print(f'\n✅ Prosody extracted for {len(prosody_data)} videos')
print(f'   Failed (no audio): {len(failed)}')
print(f'   Already extracted: {skipped_already}')
print(f'   Time: {(time.time()-t0)/60:.1f} min')

## Step 10: Save combined embeddings as .npz for reuse

In [ ]:
# Save combined embeddings as .npz for reuse
import torch

def prepare_combined_data(data_list, prosody_dict):
    """Combine WavLM + prosody + labels."""
    embeddings = []
    prosody_features = []
    labels = []
    
    for d in data_list:
        vid = d.get('video_id')
        if not vid:
            # Match by start/end
            for v in wavlm_data:
                for idx, emb in enumerate(wavlm_data[v]):
                    if abs(emb['start'] - d['start']) < 0.01 and abs(emb['end'] - d['end']) < 0.01:
                        vid = v
                        break
        
        if vid and vid in prosody_dict:
            # Find index
            for idx, emb in enumerate(wavlm_data.get(vid, [])):
                if abs(emb['start'] - d['start']) < 0.01 and abs(emb['end'] - d['end']) < 0.01:
                    embeddings.append(emb['embedding'])
                    prosody_features.append(prosody_dict[vid][idx])
                    labels.append(d['label'])
                    break
    
    return np.array(embeddings, dtype=np.float32), np.array(prosody_features, dtype=np.float32), np.array(labels, dtype=np.int64)

print('💾 Preparing combined datasets...')
train_emb, train_pros, train_lbl = prepare_combined_data(train_data, prosody_data)
val_emb, val_pros, val_lbl = prepare_combined_data(val_data, prosody_data)
test_emb, test_pros, test_lbl = prepare_combined_data(test_data, prosody_data)

print(f'✅ Train: {len(train_emb)} samples')
print(f'   Val: {len(val_emb)} samples')
print(f'   Test: {len(test_emb)} samples')

# Save to Drive
OUTPUT_DIR = Path('/content/gdrive/MyDrive/wavlm_prosody_combined_v16')
OUTPUT_DIR.mkdir(exist_ok=True)

np.savez_compressed(OUTPUT_DIR / 'train.npz',
                   embeddings=train_emb, prosody=train_pros, labels=train_lbl)
np.savez_compressed(OUTPUT_DIR / 'val.npz',
                   embeddings=val_emb, prosody=val_pros, labels=val_lbl)
np.savez_compressed(OUTPUT_DIR / 'test.npz',
                   embeddings=test_emb, prosody=test_pros, labels=test_lbl)

print(f'\n✅ Saved combined embeddings to {OUTPUT_DIR}')
print(f'   Train shape: {train_emb.shape} (WavLM) + {train_pros.shape} (Prosody)')
print(f'   Val shape: {val_emb.shape} (WavLM) + {val_pros.shape} (Prosody)')
print(f'   Test shape: {test_emb.shape} (WavLM) + {test_pros.shape} (Prosody)')

## Step 11: Train fusion model (FIXED ARCHITECTURE!)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import f1_score, classification_report
    # FIXED: Use fixed class weights [1.0, 2.5] per CURRENT_STATUS.md
    # These are the VALIDATED weights from the working Kaggle pipeline

# Convert to tensors
train_emb_t = torch.tensor(train_emb, dtype=torch.float32)
train_pros_t = torch.tensor(train_pros, dtype=torch.float32)
train_lbl_t = torch.tensor(train_lbl, dtype=torch.long)

val_emb_t = torch.tensor(val_emb, dtype=torch.float32)
val_pros_t = torch.tensor(val_pros, dtype=torch.float32)
val_lbl_t = torch.tensor(val_lbl, dtype=torch.long)

test_emb_t = torch.tensor(test_emb, dtype=torch.float32)
test_pros_t = torch.tensor(test_pros, dtype=torch.float32)
test_lbl_t = torch.tensor(test_lbl, dtype=torch.long)

# Create datasets
train_ds = TensorDataset(train_emb_t, train_pros_t, train_lbl_t)
val_ds = TensorDataset(val_emb_t, val_pros_t, val_lbl_t)
test_ds = TensorDataset(test_emb_t, test_pros_t, test_lbl_t)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)

print(f'📊 DataLoaders created')
print(f'   Train: {len(train_ds)} batches')
print(f'   Val: {len(val_ds)} batches')
print(f'   Test: {len(test_ds)} batches')

In [ ]:
import torch.nn as nn

class FusionModel(nn.Module):
    """Proper 2-class fusion model matching current best practices."""
    def __init__(self, wavlm_dim=768, prosody_dim=21):
        super().__init__()
        
        # Prosody projection (21 -> 32)
        self.prosody_proj = nn.Sequential(
            nn.Linear(prosody_dim, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32)
        )
        
        # Fusion: WavLM (768) + Prosody (32) = 800
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(wavlm_dim + 32, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, 2)  # 2-class output (no sigmoid!)
        )
    
    def forward(self, wavlm_emb, prosody):
        prosody_feat = self.prosody_proj(prosody)
        x = torch.cat([wavlm_emb, prosody_feat], dim=-1)  # 800-dim
        logits = self.classifier(x)  # 2-class logits
        return logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Device: {device}')

model = FusionModel().to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f'📊 Total parameters: {total_params:,}')

In [ ]:
# Compute class weights for balanced training
all_labels = train_lbl.tolist()
    # FIXED: Use fixed class weights [1.0, 2.5] per CURRENT_STATUS.md
    # These are the VALIDATED weights from the working Kaggle pipeline
    class_weights = torch.tensor([1.0, 2.5], dtype=torch.float32).to(device)
print(f'⚖️ Class weights: {class_weights}')

# CrossEntropyLoss with class weights (FIXED from v15!)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS = 20
best_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    t0 = time.time()
    
    # Train
    model.train()
    train_loss = 0
    for wavlm_b, prosody_b, labels_b in train_loader:
        wavlm_b = wavlm_b.to(device)
        prosody_b = prosody_b.to(device)
        labels_b = labels_b.to(device)
        
        optimizer.zero_grad()
        logits = model(wavlm_b, prosody_b)  # 2-class logits
        loss = criterion(logits, labels_b)
        loss.backward()
        optimizer.step()
        # FIXED: Add gradient clipping (per CURRENT_STATUS.md)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        train_loss += loss.item()
    
    scheduler.step()
    
    # Validate
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for wavlm_b, prosody_b, labels_b in val_loader:
            wavlm_b = wavlm_b.to(device)
            prosody_b = prosody_b.to(device)
            
            logits = model(wavlm_b, prosody_b)
            preds = torch.argmax(logits, dim=-1)  # Take argmax for 2-class
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels_b.numpy())
    
    val_f1 = f1_score(val_labels, val_preds, average='binary')
    
    epoch_time = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f} | Time: {epoch_time:.1f}s')
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = model.state_dict().copy()
        torch.save(best_state, str(OUTPUT_DIR / 'best_fusion_model_v16.pt'))
        print(f'  ✅ New best!')

In [ ]:
# Final evaluation
model.load_state_dict(best_state)
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for wavlm_b, prosody_b, labels_b in test_loader:
        wavlm_b = wavlm_b.to(device)
        prosody_b = prosody_b.to(device)
        
        logits = model(wavlm_b, prosody_b)
        preds = torch.argmax(logits, dim=-1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels_b.numpy())

test_f1 = f1_score(test_labels, test_preds, average='binary')
print(f'🏆 Test F1: {test_f1:.4f}')
print()
print(classification_report(test_labels, test_preds, target_names=['No Laughter', 'Laughter']))